# Explore relationships in real observations

Install welt-client with its parquet extra, then configure WELT_API_KEY locally. Run the cells in order. These are real hosted requests: discovery retains a private observational dataset, job and native graph/score result. No extra helper file is needed. Outputs are deliberately not saved in this source.

Iris supplies real plant measurements, not a true causal graph. This exercise asks what native structure the selected discovery model proposes. It cannot establish causal truth or measure graph accuracy. The encoded species label is deliberately excluded; four continuous observations and all150 rows are retained. No supervised train/test recipe or classification score applies.

In [ ]:
from pathlib import Path
from tempfile import mkdtemp
import pandas as pd
from welt import Client, CausalDiscovery

# 1. Get a real table from Welt's research inventory.
dataset_id = "asset-d839644b6de36fc5cafdda18c5daeca3"
version = "d839644b6de36fc5cafdda18c5daeca3284a5ce8bb6169aff2fcb3fbb4a12021"
path = Path(mkdtemp(prefix="welt-iris-")) / "iris.parquet"
with Client() as client:
    info = client.research_dataset(dataset_id)
    print("Dataset:", info["name"], "— License:", info["license"])
    for notice in info["license_notices"]:
        print(notice)
    client.download_research_dataset(dataset_id, path, version=version)
data = pd.read_parquet(path)
target = "__target__#0"  # The target recorded in this research snapshot.

# 2. Look at the rows before modelling.
print(data.head())
print("Rows and columns:", data.shape)
data.info()
print(data[target].value_counts())



In [ ]:
# Use the four measurements as observations, not the encoded class label.
# Retain all 150 rows and all four continuous measurement columns.
observations = data.drop(columns=target)
print(observations.head())
print("Observation shape:", observations.shape)
with Client() as client:
    entry = next(item for item in client.models() if item["id"] == "arrow")
    profile = next(item for item in entry["task_profiles"]
                   if item["task"] == "causal_discovery")
    if profile["status"] != "available" or not profile["execution_available"]:
        raise RuntimeError("Arrow discovery is not available; inspect the model catalogue.")
result = CausalDiscovery(model="arrow", random_state=9).discover(observations)
print("Native graph type:", result.native_graph_type)
print("Variables:", result.variables)
print("Native edges:", result.edge_marks)
print("Assumptions:", result.assumptions)
print("Model version:", result.model_version)
with Client() as client:
    scores = client.causal_scores(result.job_id)
    reopened = client.causal_result(result.job_id)
print("Score semantics:", scores["score_semantics"])
print("Retained graph identity:", reopened.id == result.id)


Inspect the native edges, assumptions and model version. Native edge probabilities are not calibrated confidence or effect estimates. Arrow preserves its declared DAG decoder; CDFM/AVICI preserve their own directed outputs and may contain cycles. Local Ergodic conversion is separately gated on the approved co-release distribution, which is not yet publicly installable. Do not install the unrelated package-index project or project a graph into a DAG. Primary graphs and scores remain retained until an explicit owned deletion; this notebook deletes nothing.